[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/01_quick_start.ipynb)

# 01: Quick Start: Consensus Trait Inference on Pride & Prejudice

> **Demonstrates the Experiment 1 headline: multi-provider consensus trait inference.**
> - **RQ1:** *Can multi-provider LLM consensus reliably infer OCEAN personality traits from literary text?*
> - **RQ2:** *Does consensus improve over the best individual provider?*

End-to-end consensus-inference demonstration. We:

1. Fetch the companion data tarball (evidence packs + ground truth for Pride & Prejudice)
2. Load evidence for a handful of main characters
3. Run LLM consensus across OpenAI + Anthropic + Google
4. Score the consensus against ground truth (PA, TD, CMC)

**Runtime:** ~10 minutes on a free Colab instance.

**Cost:** under $0.10 with default Haiku/Flash-tier models.

**API keys required:** `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. In Colab, add them as secrets; locally, drop them into a `.env` file.

> **No API keys? It still runs.** With no provider keys set, section 3 falls back to an open-weight model (Qwen2.5-1.5B-Instruct) answering the same probe prompt locally: a demo-only single rater, not one of the paper's methods. With keys, you get the multi-provider consensus (M4). The $0 headline reproducers remain notebooks 03 and 09.

## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [1]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

## 0. Environment setup

Install the companion repo's dependencies and make the `pillar1` module importable.

In [2]:
# In Colab, uncomment:
# !git clone https://github.com/Wildertrek/catcher-in-the-cache.git
# %cd catcher-in-the-cache
# !pip install -q -r requirements.txt

import sys
from pathlib import Path

# Locally: assume the notebook lives in notebooks/ inside the repo.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

Repo root: /Users/jsr/Documents/GitHub/catcher-in-the-cache


In [3]:
# API keys are OPTIONAL. With no keys, the notebook falls back to a small
# open-weight model so anyone (including ACM reviewers) can run the live demo.
# In Colab with keys, uncomment:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

import os
KEYS = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
have = [k for k in KEYS if os.getenv(k)]
for k in KEYS:
    print(f"  {k}: {'set' if os.getenv(k) else 'not set'}")

MODE = "live-consensus" if have else "open-weight"
if MODE == "live-consensus":
    print("\nMODE = live-consensus (multi-provider, the paper's M4 pipeline)")
else:
    print("\nMODE = open-weight (no keys found)")
    print("  Falls back to Qwen/Qwen2.5-1.5B-Instruct via transformers: the SAME")
    print("  probe prompt, answered locally. Demo-only single rater; it is NOT")
    print("  one of the paper's methods (M1-M6) and its scores are illustrative.")
    print("  A GPU runtime is recommended in Colab (Runtime > Change runtime type).")

  OPENAI_API_KEY: set
  ANTHROPIC_API_KEY: set
  GOOGLE_API_KEY: set

MODE = live-consensus (multi-provider, the paper's M4 pipeline)


## 1. Locate the data

The companion ships the ground-truth and evidence-pack data in-repo (under `data/`), so no download is needed. The cells below load it directly.

In [4]:
# The data ships in-repo under data/aperture-data-v1/
DATA_ROOT = ROOT / "data" / "aperture-data-v1"
if not DATA_ROOT.exists():
    DATA_ROOT = ROOT / "data"   # fallback: paths already flattened into data/

GT_PATH = DATA_ROOT / "ground_truth" / "pride_and_prejudice.json"
EVIDENCE_DIR = DATA_ROOT / "indices" / "pride_and_prejudice" / "evidence_packs"

for p in [GT_PATH, EVIDENCE_DIR]:
    status = "ok" if p.exists() else "missing"
    print(f"  {p}: {status}")

  /Users/jsr/Documents/GitHub/catcher-in-the-cache/data/aperture-data-v1/ground_truth/pride_and_prejudice.json: ok
  /Users/jsr/Documents/GitHub/catcher-in-the-cache/data/aperture-data-v1/indices/pride_and_prejudice/evidence_packs: ok


## 2. Load evidence for a few main characters

Pride & Prejudice has ~60 characters in the full registry. For the quick start we run consensus on five mains and report per-character metrics. Full-book reproduction is a one-line change, just drop the `MAIN_CHARACTERS` filter.

In [5]:
import json

# BookNLP canonical names. "Elizabeth Bennet" is stored as "Elizabeth"; likewise for Jane and Lydia.
MAIN_CHARACTERS = [
    "Elizabeth",
    "Mr. Darcy",
    "Jane",
    "Mr. Bennet",
    "Mr. Collins",
]

def load_evidence_packs(evidence_dir: Path, names: list[str]) -> list[dict]:
    packs = []
    for f in evidence_dir.glob("*.json"):
        with open(f) as fh:
            pack = json.load(fh)
        if pack.get("character_name") in names and pack.get("quote_count", 0) >= 5:
            packs.append(pack)
    return packs

packs = load_evidence_packs(EVIDENCE_DIR, MAIN_CHARACTERS)
print(f"Loaded {len(packs)} evidence packs:")
for p in packs:
    print(f"  {p['character_name']:<20} quotes={p['quote_count']:<4} coref_id={p['coref_id']}")

Loaded 5 evidence packs:
  Mr. Collins          quotes=55   coref_id=103
  Mr. Bennet           quotes=76   coref_id=118
  Mr. Darcy            quotes=91   coref_id=127
  Elizabeth            quotes=437  coref_id=107
  Jane                 quotes=137  coref_id=192


## 3. Run consensus

Three providers, median aggregation. On Haiku/Flash tier models this runs in a few minutes.

In [6]:
from pillar1.consensus_runner import run_consensus, build_prompt, aggregate_vectors

if MODE == "live-consensus":
    run = run_consensus(
        evidence_packs=packs,
        providers=["openai", "anthropic", "google"],
        aggregation="median",
        out_path=ROOT / "data" / "runs" / "pride_and_prejudice_quickstart.json",
    )
else:
    # Open-weight fallback: same probe prompt, one open model, no keys needed.
    import json as _json, re as _re, uuid as _uuid
    from transformers import AutoModelForCausalLM, AutoTokenizer
    OW_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"Loading {OW_MODEL} (first run downloads ~3 GB)...")
    _tok = AutoTokenizer.from_pretrained(OW_MODEL)
    _model = AutoModelForCausalLM.from_pretrained(OW_MODEL, torch_dtype="auto", device_map="auto")

    def open_weight_rate(prompt: str):
        text = _tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True)
        ids = _tok([text], return_tensors="pt").to(_model.device)
        out = _model.generate(**ids, max_new_tokens=300, do_sample=False,
                              pad_token_id=_tok.eos_token_id)
        resp = _tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        m = _re.search(r"\{.*?\}", resp, _re.S)
        if not m:
            return None, resp
        try:
            parsed = _json.loads(m.group(0))
            vec = {k: max(-1.0, min(1.0, float(parsed[k]))) for k in "OCEAN" if k in parsed}
        except Exception:
            return None, resp
        return (vec if len(vec) == 5 else None), resp

    characters = []
    for pack in packs:
        vec, raw_text = open_weight_rate(build_prompt(pack))
        ok = vec is not None
        print(f"  {pack['character_name']}: {'ok' if ok else 'PARSE FAILED'}")
        characters.append({
            "character_name": pack["character_name"],
            "coref_id": pack.get("coref_id"),
            "vectors": ([{"provider": "open-weight", "model": OW_MODEL,
                          "vector": vec, "raw": raw_text[:500]}] if ok else []),
            "consensus": vec,
        })
    run = {
        "run_id": f"quickstart-openweight-{_uuid.uuid4().hex[:8]}",
        "providers": ["open-weight"], "models": {"open-weight": OW_MODEL},
        "aggregation": "single-rater (open-weight fallback)",
        "characters": characters,
    }

print(f"Run ID: {run['run_id']}")
print(f"Characters scored: {sum(1 for c in run['characters'] if c['consensus'])}/{len(run['characters'])}")

Run ID: run_20260722T210108Z
Characters scored: 5/5


In [7]:
# Preview consensus vectors
for char in run["characters"]:
    cons = char["consensus"]
    if cons:
        vec = " ".join(f"{t}={cons[t]:+.2f}" for t in "OCEAN")
        print(f"  {char['character_name']:<20} {vec}")
    else:
        print(f"  {char['character_name']:<20} NO CONSENSUS (provider failures)")

  Mr. Collins          O=-0.50 C=+0.50 E=+0.20 A=+0.30 N=+0.40
  Mr. Bennet           O=+0.20 C=+0.10 E=+0.30 A=+0.00 N=+0.10
  Mr. Darcy            O=+0.50 C=+0.70 E=-0.30 A=+0.20 N=+0.40
  Elizabeth            O=+0.70 C=+0.50 E=+0.60 A=+0.80 N=+0.30
  Jane                 O=+0.50 C=+0.70 E=+0.30 A=+0.80 N=+0.20


## 4. Score against ground truth

- **PA (MAE):** lower is better
- **PA (Pearson):** higher is better
- **TD:** mean pairwise Euclidean distance across consensus vectors (higher = characters more distinguishable)
- **CMC:** `1 - mean(across-provider variance)` (higher = providers agree)

In [8]:
from pillar1.metrics import (
    load_ground_truth, compute_pa_mae, compute_pa_pearson,
    compute_td, compute_cmc,
)

gt = load_ground_truth(GT_PATH)
print(f"Ground truth loaded: {len(gt)} characters with OCEAN scores\n")

print(f"{'Character':<20} {'PA-MAE':>8} {'PA-r':>8}")
print("-" * 38)
for char in run["characters"]:
    cons = char["consensus"]
    key = char.get("coref_id") or char["character_name"]
    schol = gt.get(key) or gt.get(char["character_name"])
    if cons and schol:
        mae, _ = compute_pa_mae(cons, schol)
        r = compute_pa_pearson(cons, schol)
        r_str = f"{r:+.3f}" if r is not None else "  n/a"
        print(f"{char['character_name']:<20} {mae:>8.3f} {r_str:>8}")

Ground truth loaded: 18 characters with OCEAN scores

Character              PA-MAE     PA-r
--------------------------------------
Mr. Collins             0.309   +0.590
Mr. Bennet              0.345   +0.262
Mr. Darcy               0.184   +0.786
Elizabeth               0.259   +0.614
Jane                    0.307   +0.300


In [9]:
# TD across the scored characters
scored = [(i, c["character_name"], c["consensus"])
          for i, c in enumerate(run["characters"]) if c["consensus"]]
td = compute_td(scored)
print(f"TD mean: {td['mean']:.3f}  (n_pairs={td['n_pairs']})")
if td["n_pairs"] > 0:
    print(f"  Most similar : {td['most_similar'][0][0]} <-> {td['most_similar'][0][1]} (d={td['most_similar'][0][2]:.3f})")
    print(f"  Most distinct: {td['most_distinct'][-1][0]} <-> {td['most_distinct'][-1][1]} (d={td['most_distinct'][-1][2]:.3f})")
else:
    print("  (need >=2 characters with consensus for pairwise distinctiveness -- set API keys and re-run)")

TD mean: 1.010  (n_pairs=10)
  Most similar : Elizabeth <-> Jane (d=0.424)
  Most distinct: Mr. Collins <-> Elizabeth (d=1.364)


In [10]:
# CMC per character (requires >= 2 successful provider vectors)
cmc = compute_cmc(run["characters"])
for name, r in cmc.items():
    print(f"  {name:<20} CMC={r['cmc']:.3f}  (n_providers={r['n_providers']})")

## Expected results (reference)

Running on the full 29-novel corpus gives **CvG MAE ≈ 0.298** [CI 0.274, 0.324] against curated ground truth and **Pearson r ≈ 0.635** [CI 0.567, 0.699]. The five-character quick-start run shown above typically lands within ±0.05 of those figures; wider variation is expected on the small sample.

See `docs/reproducibility.md` for tolerances and `pillar1/README.md` for a full description of each metric.

## Next steps

- **More characters:** remove the `MAIN_CHARACTERS` filter to run on every character in the evidence-packs directory.
- **Regressor inference:** see `notebooks/10_regressor_inference.ipynb` for the distilled trait predictor (no LLM calls required).